# Prep 2 · Images, Fourier transforms, and k-space

**Time:** about 90 minutes. **Needs:** Prep 1. No patient data — we use a synthetic phantom.

An MRI scanner measures the **Fourier transform** of the image (called *k-space*), not the image. This
notebook builds the intuition you need in three moves — 1-D signals, 2-D images, undersampling — and ends
with the repo's mask and forward-operator milestones (`3/8 → 8/8`).

In [ ]:
import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt
from skimage.data import shepp_logan_phantom
from skimage.transform import resize

from mrigen.fourier import fft2c, ifft2c     # the repo's ONE Fourier convention (given)

## 1. One dimension: a signal is a sum of frequencies

The Fourier transform rewrites a signal as a sum of sines and cosines. Low frequencies are slow trends;
high frequencies are fine wiggles. `fftshift` just moves frequency zero to the middle of the array so the
plot reads "low in the centre, high at the edges" — the same convention the repo uses for k-space.

In [ ]:
n = 128
t = np.arange(n) / n
signal = 1.0 * np.sin(2 * np.pi * 2 * t) + 0.4 * np.sin(2 * np.pi * 15 * t)   # slow + fast component

spectrum = np.fft.fftshift(np.fft.fft(signal, norm="ortho"))
freqs = np.fft.fftshift(np.fft.fftfreq(n, d=1 / n))

fig, ax = plt.subplots(1, 2, figsize=(11, 3))
ax[0].plot(t, signal); ax[0].set_title("signal")
ax[1].stem(freqs, np.abs(spectrum), basefmt=" "); ax[1].set_title("|spectrum| (zero frequency in the centre)")
ax[1].set_xlim(-25, 25); plt.tight_layout(); plt.show()

In [ ]:
# Keep only the low frequencies (|f| < 6): the fast wiggle disappears, the slow trend survives.
keep_low = np.abs(freqs) < 6
low_only = np.fft.ifft(np.fft.ifftshift(spectrum * keep_low), norm="ortho").real
high_only = np.fft.ifft(np.fft.ifftshift(spectrum * ~keep_low), norm="ortho").real

fig, ax = plt.subplots(1, 2, figsize=(11, 3))
ax[0].plot(t, signal, alpha=.4, label="signal"); ax[0].plot(t, low_only, label="low frequencies only"); ax[0].legend()
ax[1].plot(t, signal, alpha=.4, label="signal"); ax[1].plot(t, high_only, label="high frequencies only"); ax[1].legend()
plt.tight_layout(); plt.show()

## 2. Two dimensions: an image and its k-space

Same idea, two axes. The repo's `fft2c` / `ifft2c` are the 2-D transform with the zero frequency
**centred** and an **orthonormal** scaling (so energy is the same in both domains). They are given —
never write your own — because a stray shift or scale factor is the number-one silent bug in MRI code.

We use the Shepp–Logan phantom, the classic synthetic "head", so nothing here needs the fastMRI data.

In [ ]:
x = resize(shepp_logan_phantom(), (128, 128), anti_aliasing=True).astype(np.float32)
x = x / x.max()                          # the repo's convention: images live in [0, 1]
k = fft2c(jnp.asarray(x))                # complex64 k-space, DC in the centre

fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(x, cmap="gray"); ax[0].set_title("image  x"); ax[0].axis("off")
ax[1].imshow(np.log1p(np.abs(np.asarray(k))), cmap="viridis"); ax[1].set_title("k-space  log|k|"); ax[1].axis("off")
plt.show()
print("round-trip error:", float(jnp.abs(ifft2c(k).real - x).max()))

### Exercise 1 — what the centre and the edges of k-space carry

Make two images from `k`: one from **only the central 16 × 16 block** of k-space (everything else zero),
and one from **everything except** that block. Store the real parts as `centre_img` and `edges_img`.
Look at them: which one has the contrast, which one has the edges?

In [ ]:
c = 8
H, W = k.shape
block = jnp.zeros_like(k).at[H//2 - c:H//2 + c, W//2 - c:W//2 + c].set(1.0)   # 1 inside the central block

centre_img = ifft2c(k * block).real
edges_img = ifft2c(k * (1 - block)).real

<details><summary><b>Hint</b> — try the exercise first, then click to reveal</summary>

Build the block the same way as the masks in Prep 1: `jnp.zeros_like(k).at[...].set(1.0)` with the
slices `H//2 - c : H//2 + c` on both axes. Multiplying k-space by `block` keeps only the centre,
by `(1 - block)` keeps only the edges — inverse-transform each product and take `.real`.

</details>

In [ ]:
assert centre_img is not ... and edges_img is not ..., "fill in the exercise above first"
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
for a, img, title in zip(ax, (x, centre_img, edges_img), ("image", "centre of k-space only", "edges of k-space only")):
    a.imshow(np.asarray(img), cmap="gray"); a.set_title(title); a.axis("off")
plt.show()

# check: the centre carries most of the energy (contrast); the edges carry the detail
rel = lambda a: float(jnp.linalg.norm(jnp.asarray(a) - x) / jnp.linalg.norm(x))
assert centre_img.shape == edges_img.shape == (128, 128)
assert rel(centre_img) < rel(edges_img), "the centre-only image should be closer to x than the edges-only one"
assert abs(float(jnp.sum(jnp.abs(k) ** 2)) - float(jnp.sum(x ** 2))) < 1e-2 * float(jnp.sum(x ** 2)), "Parseval: energy is preserved"
print("exercise 1 OK ·  relative error: centre-only", round(rel(centre_img), 3), "· edges-only", round(rel(edges_img), 3))

## 3. Accelerating the scan = skipping columns of k-space

A Cartesian scan acquires k-space one column (one *phase-encode line*) at a time, so the scan gets faster
by acquiring fewer columns. A binary **mask** `M` records which columns were measured. We always keep a
fully sampled band of central columns — the **ACS** region — because the centre carries most of the energy.

Then the naive image is `ifft2c(M ⊙ k)`: zeros where we did not measure. It is called the **zero-filled**
reconstruction, and it is full of **aliasing** — ghost copies of the anatomy. Look:

In [ ]:
def equispaced_mask_np(shape, R, acs_frac=0.08):
    """Every R-th column on, plus a central band of acs_frac*W columns. (Plain NumPy, for this notebook.)"""
    H, W = shape
    m = np.zeros((H, W), np.float32)
    m[:, ::R] = 1.0
    n_acs = max(1, int(round(acs_frac * W))); start = (W - n_acs) // 2
    m[:, start:start + n_acs] = 1.0
    return m

fig, ax = plt.subplots(2, 3, figsize=(12, 8))
for row, R in zip(ax, (4, 8)):
    M = equispaced_mask_np((128, 128), R)
    zf = ifft2c(M * k).real
    row[0].imshow(x, cmap="gray", vmin=0, vmax=1); row[0].set_title("fully sampled")
    row[1].imshow(M, cmap="gray"); row[1].set_title(f"mask, R = {R}  (white = measured)")
    row[2].imshow(np.asarray(zf), cmap="gray", vmin=0, vmax=1); row[2].set_title(f"zero-filled, R = {R}")
    for a in row: a.axis("off")
plt.tight_layout(); plt.show()

### Exercise 2 — the effective acceleration

The *nominal* acceleration is `R`, but the ACS band adds columns, so the fraction actually measured is a
bit more than `1/R`. The **effective acceleration** `R_eff` is the reciprocal of the fraction of k-space
actually measured. Compute it for the `R = 4` mask above. Why is it below 4?

In [ ]:
M4 = equispaced_mask_np((128, 128), 4)
R_eff = M4.size / M4.sum()

<details><summary><b>Hint</b> — try the exercise first, then click to reveal</summary>

The fraction measured is `M4.sum() / M4.size` (the mask is 1 where a column was acquired), so its
reciprocal is `M4.size / M4.sum()`. This is the same formula the repo uses everywhere.

</details>

In [ ]:
# check
assert R_eff is not ..., "fill in the exercise above first"
print(f"nominal R = 4, effective R = {R_eff:.2f}  (measured {M4.mean():.1%} of k-space)")
assert 3.0 < R_eff < 4.0, "with an ACS band the effective acceleration is a little below the nominal one"
print("exercise 2 OK")

## 4. Bridge to the repo — six milestones

Two files, both short, both tested:

**`src/mrigen/masks.py`** — `equispaced_mask` and `random_mask`. Same idea as `equispaced_mask_np`
above, but returning a JAX array and using the given `acs_columns` helper. For `random_mask`, keep the
ACS band and add randomly chosen columns until the *total* fraction kept is about `1/R`.

**`src/mrigen/recon/operators.py`** — the measurement model, three one-liners built from `fft2c`,
`ifft2c` and the mask:

- `forward(x, mask)` — image → undersampled k-space: this is `A(x)`, exactly what the scanner does;
- `adjoint(k, mask)` — k-space → image (the zero-filled recon is `adjoint(y).real`);
- `data_consistency(x_est, y_obs, mask)` — keep the measured k-space, trust the estimate elsewhere.

You have already done all of this in this notebook — the cell under section 3 *is* the forward model
and the zero-filled recon. Work each function out from what it must do to k-space (the docstrings in
the file pin down the exact conventions).

Implement them **one function at a time**: each has its own check cell below, independent of the
others, and its own test to run in a terminal. While a function still raises `NotImplementedError`
its test reports itself as *skipped*; once your implementation is in, the same command reports
*passed* — that flip is your signal.

#### `masks.py`, function by function

In [ ]:
# check equispaced_mask on its own
import importlib
import mrigen.masks
importlib.reload(mrigen.masks)            # picks up your edit without restarting the kernel
from mrigen.masks import equispaced_mask

try:
    M = equispaced_mask((128, 128), acceleration=4)
    assert M.shape == (128, 128) and bool(jnp.all((M == 0) | (M == 1)))
    assert bool(jnp.all(M == M[0][None, :])), "whole columns on or off"
    plt.figure(figsize=(3, 3)); plt.imshow(np.asarray(M), cmap="gray"); plt.title("equispaced_mask, R=4"); plt.axis("off"); plt.show()
    print("equispaced_mask OK - confirm with:  pixi run test tests/test_masks.py -k equispaced")
except NotImplementedError as e:
    print("Not yet:", e, "-> implement it in src/mrigen/masks.py, then re-run this cell")

In [ ]:
# check random_mask on its own
importlib.reload(mrigen.masks)
from mrigen.masks import random_mask

try:
    Mr = random_mask((128, 128), acceleration=4, seed=0)
    assert Mr.shape == (128, 128) and bool(jnp.all((Mr == 0) | (Mr == 1)))
    assert 0.2 < float(Mr.mean()) < 0.35, "should keep about 1/4 of the columns"
    plt.figure(figsize=(3, 3)); plt.imshow(np.asarray(Mr), cmap="gray"); plt.title("random_mask, R=4"); plt.axis("off"); plt.show()
    print("random_mask OK - confirm with:  pixi run test tests/test_masks.py -k random")
except NotImplementedError as e:
    print("Not yet:", e, "-> implement it in src/mrigen/masks.py, then re-run this cell")

#### `operators.py`, function by function

These checks use the notebook's own `equispaced_mask_np` and the phantom's k-space `k` from section 2,
so they work even before your `masks.py` functions do.

In [ ]:
# check forward on its own
import mrigen.recon.operators
importlib.reload(mrigen.recon.operators)
from mrigen.recon import operators as op

Mnp = jnp.asarray(equispaced_mask_np((128, 128), 4))
try:
    y = op.forward(jnp.asarray(x), Mnp)
    assert y.shape == (128, 128) and jnp.iscomplexobj(y)
    assert bool(jnp.allclose(y[Mnp == 0], 0.0)), "unmeasured entries must be exactly zero"
    assert float(jnp.abs(y).max()) > 0, "measured entries must carry the k-space"
    print("forward OK - confirm with:  pixi run test tests/test_operators.py -k forward")
except NotImplementedError as e:
    print("Not yet:", e, "-> implement it in src/mrigen/recon/operators.py, then re-run this cell")

In [ ]:
# check adjoint on its own
importlib.reload(mrigen.recon.operators)
from mrigen.recon import operators as op

Mnp = jnp.asarray(equispaced_mask_np((128, 128), 4))
try:
    assert bool(jnp.allclose(op.adjoint(k, jnp.ones_like(Mnp)).real, x, atol=1e-4)), \
        "with a full mask, adjoint must invert the FFT exactly"
    zf = op.adjoint(Mnp * k, Mnp).real
    fig, ax = plt.subplots(1, 2, figsize=(7, 3.5))
    ax[0].imshow(x, cmap="gray"); ax[0].set_title("image"); ax[0].axis("off")
    ax[1].imshow(np.asarray(zf), cmap="gray"); ax[1].set_title("zero-filled via adjoint"); ax[1].axis("off")
    plt.show()
    print("adjoint OK (no dedicated unit test - the full suite's round-trip tests cover it)")
except NotImplementedError as e:
    print("Not yet:", e, "-> implement it in src/mrigen/recon/operators.py, then re-run this cell")

In [ ]:
# check data_consistency on its own
importlib.reload(mrigen.recon.operators)
from mrigen.recon import operators as op

Mnp = jnp.asarray(equispaced_mask_np((128, 128), 4))
try:
    dc = op.data_consistency(jnp.asarray(x), Mnp * k, Mnp)
    assert bool(jnp.allclose(dc, x, atol=1e-4)), "DC of the truth, given the truth's k-space, must return the truth"
    print("data_consistency OK - confirm with:  pixi run test tests/test_operators.py -k consistency")
except NotImplementedError as e:
    print("Not yet:", e, "-> implement it in src/mrigen/recon/operators.py, then re-run this cell")

When all five checks print OK, run the whole suite — `pixi run test`, no more skips from these files —
and `pixi run milestones` should say **8/8**.

## Optional: the real thing

Once your fastMRI registration has come through, follow `data/REGISTER_FIRST.md`, run
`pixi run download` and `pixi run preprocess`, and open `notebooks/00_data_and_kspace.ipynb`: the same
pictures as above, on a real knee.

## Done when

- every check cell prints OK and `pixi run milestones` says `8/8`;
- you can explain, in a sentence each: what k-space is, why the centre is kept, what aliasing looks like.

Next: **Prep 3 · Bayesian inference by hand.**